In [ ]:
but :
https://3.6.15.198:7082/api/list-premium-posts
Error: write EPROTO 1023040:error:100000f7:SSL routines:OPENSSL_internal:WRONG_VERSION_NUMBER:..\..\..\..\src\third_party\boringssl\src\ssl\tls_record.cc:231:

In [ ]:
The core issue here is a mixed content problem due to the mismatch between HTTP and HTTPS. Let me break down the situation, explain why the error is occurring, and provide a solution to allow both HTTP and HTTPS while ensuring your site (https://adtip.in) can communicate with your backend (3.6.15.198:7082) and resolve the NS_ERROR_GENERATE_FAILURE and EPROTO errors.

In [ ]:
Why the Error is Happening

    Mixed Content Issue:
        Your frontend is hosted on https://adtip.in (HTTPS), but your backend API calls are configured to use http://3.6.15.198:7082/api/list-premium-posts (HTTP) in local development.
        When the site is hosted, the browser enforces HTTPS for all requests from an HTTPS page due to security policies. This causes the browser to block or fail HTTP requests, resulting in the NS_ERROR_GENERATE_FAILURE seen in the Network tab.
    HTTPS on Backend Fails:
        When you try to access https://3.6.15.198:7082/api/list-premium-posts (HTTPS), it fails with the error:
        text

    Error: write EPROTO 1023040:error:100000f7:SSL routines:OPENSSL_internal:WRONG_VERSION_NUMBER:..\..\..\..\src\third_party\boringssl\src\ssl\tls_record.cc:231:
    This EPROTO error in Postman indicates that the backend server (3.6.15.198:7082) is not properly configured for HTTPS. The WRONG_VERSION_NUMBER error typically means:
        The server isn’t set up to handle HTTPS (i.e., it’s not using the https module or doesn’t have a valid SSL certificate).
        The client (Postman or browser) is trying to negotiate an SSL/TLS connection, but the server responds incorrectly because it’s expecting HTTP, not HTTPS.

Local vs. Hosted Environment:

    In your local environment, both the frontend and backend use HTTP (e.g., http://localhost:3000 and http://3.6.15.198:7082), so there’s no mixed content issue, and everything works.
    When hosted, the frontend switches to HTTPS (https://adtip.in), but the backend API calls remain HTTP, causing the browser to block the requests. Even if you change the API calls to HTTPS (https://3.6.15.198:7082), the backend isn’t configured for HTTPS, so the request fails with the EPROTO error.

Impact on Google Indexing:

    Because the frontend can’t fetch data from the backend (due to the mixed content issue), it falls back to cached content or fails to load the updated "Adtip" site, showing the old "tip-tube-web" version instead.
    Google’s crawler faces the same issue when it visits https://adtip.in, preventing it from indexing the updated content with the correct title ("Adtip").

In [ ]:
Solution: Allow Both HTTP and HTTPS

To resolve this, we need to ensure that the frontend can communicate with the backend, whether the backend uses HTTP or HTTPS. The ideal solution is to set up HTTPS on the backend (since the frontend is on HTTPS), but if that’s not immediately feasible, we can use a temporary workaround to allow HTTP requests from the HTTPS frontend. Here are the steps:
1. Ideal Solution: Set Up HTTPS on the Backend

Since your frontend is on HTTPS, the best long-term solution is to configure your backend (3.6.15.198:7082) to support HTTPS. This will eliminate mixed content issues and ensure compatibility.

Steps to Set Up HTTPS on the Backend:

    Install an SSL Certificate on the EC2 Instance:
        SSH into your EC2 instance:
        bash

ssh -i your-key.pem ubuntu@3.6.15.198
Install Certbot to generate a Let’s Encrypt certificate:
bash

    sudo apt update
    sudo apt install certbot python3-certbot-nginx
    sudo certbot certonly --standalone -d api.adtip.in
        This assumes you’ve set up a subdomain (api.adtip.in) pointing to 3.6.15.198 in Hostinger’s DNS settings (as recommended earlier).
        The certificate files will be generated in /etc/letsencrypt/live/api.adtip.in/.

Update Backend Code to Use HTTPS:

    Modify your backend code to use the https module instead of http:
    javascript

const https = require("https");
const fs = require("fs");
const app = require("./config/app.js");

const server = https.createServer({
  cert: fs.readFileSync("/etc/letsencrypt/live/api.adtip.in/fullchain.pem"),
  key: fs.readFileSync("/etc/letsencrypt/live/api.adtip.in/privkey.pem"),
}, app);

server.listen(process.env.SERVER_PORT, () => {
  console.log(`Server started and running on port: ${process.env.SERVER_PORT}`);
});
If you want to continue using port 7082, ensure your AWS Security Group allows inbound traffic on 7082 for HTTPS. Alternatively, switch to port 443 (standard HTTPS port) by updating process.env.SERVER_PORT in your .env file:
text

    SERVER_PORT=443

Update AWS Security Group:

    Ensure the Security Group for your EC2 instance allows inbound traffic on the port you’re using:
    text

    Type: HTTPS
    Protocol: TCP
    Port Range: 443 (or 7082 if you keep the custom port)
    Source: 0.0.0.0/0 (or 89.116.133.221/32 for the frontend IP)

Update Frontend to Use HTTPS:

    Update the API base URL in your frontend to use HTTPS:
    env

VITE_API_URL=https://api.adtip.in:7082
Or, if using port 443:
env

    VITE_API_URL=https://api.adtip.in
    Redeploy the frontend to Hostinger’s public_html directory.

Test the HTTPS Endpoint:

    Test https://api.adtip.in:7082/api/list-premium-posts in Postman or your browser. It should now return the same response you saw with HTTP:
    json

        {
          "status": true,
          "message": "Premium post lists",
          "data": [...]
        }

This approach ensures that both the frontend and backend use HTTPS, resolving the mixed content issue and eliminating the NS_ERROR_GENERATE_FAILURE and EPROTO errors.
2. Temporary Workaround: Proxy HTTP Requests Through the Frontend Server

If setting up HTTPS on the backend isn’t immediately feasible (e.g., due to delays in obtaining an SSL certificate), you can use a reverse proxy on your frontend server (89.116.133.221) to forward API requests to the backend over HTTP. This avoids the mixed content issue because the frontend will make API calls to the same HTTPS origin, and the proxy will handle the HTTP communication with the backend.

Steps to Set Up a Reverse Proxy:

    Set Up a Subdomain for API Requests:
        In Hostinger hPanel → DNS / Nameservers → DNS Records, ensure api.adtip.in points to 3.6.15.198 (for future HTTPS setup). However, for the proxy, we’ll handle requests through the frontend server.
        Alternatively, you can use a path like https://adtip.in/api to proxy requests.
    Configure a Proxy on the Frontend Server:
        Hostinger doesn’t provide direct access to Nginx or Apache for reverse proxy configuration, but you can achieve this using a script or by serving the API through the same server.
        A simpler approach is to modify your frontend to proxy API requests. If you’re using a framework like React, Vue, or Next.js, you can set up a proxy in development and production.
    Proxy in Development (Local):
        If using Create React App, add a proxy in package.json:
        json

"proxy": "http://3.6.15.198:7082"
For Vite (common with modern React apps), add a proxy in vite.config.js:
javascript

    export default defineConfig({
      server: {
        proxy: {
          "/api": {
            target: "http://3.6.15.198:7082",
            changeOrigin: true,
          },
        },
      },
    });
    This allows your local frontend to proxy requests to the backend without mixed content issues.

Proxy in Production (Hosted):

    Since Hostinger doesn’t allow direct server configuration, you can create a simple Node.js or PHP script to act as a proxy.
    Create a file like proxy.php in public_html/api/:
    php

    <?php
    header("Access-Control-Allow-Origin: https://adtip.in");
    header("Access-Control-Allow-Methods: GET, POST, PUT, DELETE, OPTIONS");
    header("Access-Control-Allow-Headers: Content-Type, Authorization");

    $url = "http://3.6.15.198:7082" . $_SERVER['REQUEST_URI'];
    $response = file_get_contents($url);
    echo $response;
    ?>
    Update your frontend to make API requests to https://adtip.in/api/list-premium-posts instead of directly to http://3.6.15.198:7082.

Update Frontend API Base URL:

    In your frontend’s .env file:
    env

        VITE_API_URL=https://adtip.in/api
        Redeploy the frontend.

This approach ensures that the frontend makes requests to https://adtip.in/api, which proxies them to http://3.6.15.198:7082, avoiding mixed content issues.
3. Allow Both HTTP and HTTPS on the Backend (Temporary)

If you want the backend to support both HTTP and HTTPS temporarily (until you fully transition to HTTPS), you can run two servers: one for HTTP and one for HTTPS.

Update Backend Code to Support Both:

    Modify your backend code to create both HTTP and HTTPS servers:
    javascript

const http = require("http");
const https = require("https");
const fs = require("fs");
const app = require("./config/app.js");

// HTTP server
const httpServer = http.createServer(app);
httpServer.listen(7082, () => {
  console.log("HTTP Server running on port 7082");
});

// HTTPS server (if certificate is available)
let httpsServer;
try {
  httpsServer = https.createServer({
    cert: fs.readFileSync("/etc/letsencrypt/live/api.adtip.in/fullchain.pem"),
    key: fs.readFileSync("/etc/letsencrypt/live/api.adtip.in/privkey.pem"),
  }, app);
  httpsServer.listen(7083, () => {
    console.log("HTTPS Server running on port 7083");
  });
} catch (err) {
  console.log("HTTPS setup failed, running HTTP only:", err.message);
}
Update the AWS Security Group to allow inbound traffic on both ports:
text
Type: Custom TCP
Protocol: TCP
Port Range: 7082
Source: 0.0.0.0/0

Type: Custom TCP
Protocol: TCP
Port Range: 7083
Source: 0.0.0.0/0
Update the frontend to use the HTTP endpoint temporarily:
env
VITE_API_URL=http://api.adtip.in:7082
Once HTTPS is fully set up, switch to:
env

    VITE_API_URL=https://api.adtip.in:7083

Note: This is a temporary solution. Running HTTP in production exposes your API to security risks (e.g., data interception). Transition to HTTPS as soon as possible.
4. Clear Caches and Test

    Clear your browser cache and test in incognito mode.
    Flush your local DNS cache:
        Windows: ipconfig /flushdns
        macOS: sudo dscacheutil -flushcache; sudo killall -HUP mDNSResponder
    Purge any CDN caches (e.g., Hostinger CDN or Cloudflare).
    Open the browser’s Network tab and confirm that the /list-premium-posts request now succeeds with a 200 OK status.

5. Request Google Re-Indexing

    Once the frontend can successfully fetch data from the backend and the new "Adtip" site loads, request re-indexing in Google Search Console:
        Go to URL Inspection → Enter https://www.adtip.in → Click Request Indexing.
    Monitor the search result for "adtip" over the next few days to confirm the title updates to "Adtip".

Immediate Action

    Step 1: Set up HTTPS on the backend by installing an SSL certificate and updating the backend code to use the https module.
    Step 2: Update the frontend to use the HTTPS backend URL (https://api.adtip.in:7082).
    Step 3 (Alternative): If HTTPS setup isn’t immediate, set up a reverse proxy on the frontend server (https://adtip.in/api) to forward requests to http://3.6.15.198:7082.
    Step 4: Clear caches, test the site, and confirm the API request succeeds.
    Step 5: Request re-indexing in Google Search Console.

The ideal solution is to fully transition to HTTPS on the backend to avoid mixed content issues and ensure security. Let me know if you need help with any specific step, like setting up the SSL certificate or configuring the proxy!